# Lab 04: Unity Catalog Functions as Agent Tools

## Overview

In this lab you will **create Unity Catalog SQL and Python functions**, expose them as agent tools via `UCFunctionToolkit`, test them in the AI Playground, and combine them with a Vector Search retrieval tool inside a LangChain agent.

| Attribute | Detail |
|---|---|
| **Exam Domain** | Application Development (30%) |
| **Key Skills** | UC SQL UDFs, UC Python UDFs, UCFunctionToolkit, tool schemas, multi-tool agents |
| **Estimated Time** | 25 minutes |
| **Estimated Cost** | $1–2 |

### What You Will Build

```
User Query
    └─► LangChain Agent
            ├─► search_arxiv_papers  (Vector Search retrieval tool)
            ├─► get_paper_metadata   (UC SQL Function — paper stats)
            └─► format_citation      (UC Python Function — APA formatter)
```

UC functions live in the catalog and are governed by Unity Catalog permissions, making them reusable across notebooks, jobs, and AI agents without duplicating logic.

In [ ]:
# Prerequisites — run once per session
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-community unitycatalog-langchain --quiet
dbutils.library.restartPython()

> **Cost tip:** These labs run on **serverless compute** by default — no cluster setup needed. You only pay per-second of actual usage. The `COST_PROFILE` below lets you choose cheaper models if you're cost-sensitive.

In [ ]:
# === Cost Profile ===
# "budget"   — cheapest models, may have lower quality (~$0.50/lab)
# "balanced" — good quality/cost ratio (~$1-2/lab)  [DEFAULT]
# "quality"  — best models, highest cost (~$3-5/lab)
COST_PROFILE = "balanced"

_LLM_ENDPOINTS = {
    "budget":   "databricks-meta-llama-3-1-8b-instruct",
    "balanced": "databricks-meta-llama-3-3-70b-instruct",
    "quality":  "databricks-meta-llama-3-3-70b-instruct",
}
LLM_ENDPOINT = _LLM_ENDPOINTS[COST_PROFILE]

# ---------------------------------------------------------------------------
# Configuration — update these values to match your environment
# ---------------------------------------------------------------------------
CATALOG = "genai_lab_guide"   # Unity Catalog catalog name
SCHEMA  = "default"           # Schema inside the catalog

print(f"Catalog : {CATALOG}")
print(f"Schema  : {SCHEMA}")

## A. Create a SQL UC Function

UC SQL functions are stored procedures written in Spark SQL that run on serverless compute and are governed by Unity Catalog.

The function below queries the `arxiv_chunks` table (created in Lab 01) to return paper-level metadata: the source file path and the number of chunks that were extracted from each paper.

**Why SQL?** Simple aggregations and lookups over Delta tables are natural fits for SQL UDFs. The function is versioned, permissioned, and discoverable in the Catalog Explorer — any principal with `EXECUTE` privilege can call it.

In [ ]:
# Create the SQL UC function
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.get_paper_metadata(paper_name STRING)
RETURNS TABLE (path STRING, chunk_count BIGINT)
COMMENT 'Return source path and total chunk count for arXiv papers matching paper_name.'
RETURN
  SELECT
    path,
    COUNT(*) AS chunk_count
  FROM {CATALOG}.{SCHEMA}.arxiv_chunks
  WHERE LOWER(path) LIKE CONCAT('%', LOWER(paper_name), '%')
  GROUP BY path
  ORDER BY chunk_count DESC
""")

print(f"Function created: {CATALOG}.{SCHEMA}.get_paper_metadata")

# Smoke-test: find papers whose path contains 'attention'
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.get_paper_metadata('attention')"))

## B. Create a Python UC Function

UC Python functions execute arbitrary Python logic server-side and return scalar values. They are ideal for transformation and formatting tasks that go beyond what SQL can express cleanly.

The function below formats an APA-style citation string from structured metadata fields. When the agent calls `format_citation`, it gets back a properly formatted reference string without needing to prompt the LLM to do string manipulation.

In [ ]:
# Create the Python UC function
spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.format_citation(
  authors STRING,
  title   STRING,
  year    INT,
  arxiv_id STRING
)
RETURNS STRING
LANGUAGE PYTHON
COMMENT 'Format an APA-style citation string from paper metadata fields.'
AS $$
def format_citation(authors, title, year, arxiv_id):
    author_list = authors.strip()
    return f"{{author_list}} ({{year}}). {{title}}. arXiv:{{arxiv_id}}."
$$
""")

print(f"Function created: {CATALOG}.{SCHEMA}.format_citation")

# Smoke-test: format a well-known paper citation
result = spark.sql(f"""
  SELECT {CATALOG}.{SCHEMA}.format_citation(
    'Vaswani, A., Shazeer, N., Parmar, N., et al.',
    'Attention Is All You Need',
    2017,
    '1706.03762'
  ) AS citation
""").collect()[0]["citation"]

print(f"Citation: {result}")

## C. Understand Tool Schemas

When an LLM uses a UC function as a tool, it relies on the function's **schema** — name, description (the `COMMENT`), and parameter names/types — to decide when and how to call it.

The `WorkspaceClient` from the Databricks SDK lets you inspect these schemas programmatically. This is the same metadata that `UCFunctionToolkit` reads to build the tool definition that is injected into the LLM's context.

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

print(f"UC functions in {CATALOG}.{SCHEMA}\n{'=' * 60}")

for fn in w.functions.list(catalog_name=CATALOG, schema_name=SCHEMA):
    print(f"\nName    : {fn.full_name}")
    print(f"Comment : {fn.comment}")
    if fn.input_params and fn.input_params.parameters:
        print("Params  :")
        for p in fn.input_params.parameters:
            print(f"  - {p.name} ({p.type_name}): {p.comment or 'no description'}")

## D. Use UC Tools with an Agent

Now we combine three tools in a single LangChain agent:

1. **`search_arxiv_papers`** — LangChain `@tool` wrapping the Vector Search retrieval function from Lab 03.
2. **`get_paper_metadata`** — UC SQL function exposed via `UCFunctionToolkit`.
3. **`format_citation`** — UC Python function exposed via `UCFunctionToolkit`.

`UCFunctionToolkit` reads the UC function schemas and builds LangChain-compatible tool objects automatically — no manual schema writing required. The LLM selects the appropriate tool(s) based on the question.

In [ ]:
from databricks.vector_search.client import VectorSearchClient
from langchain_community.chat_models import ChatDatabricks
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_tool_calling_agent, AgentExecutor
from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from unitycatalog.ai.langchain.toolkit import UCFunctionToolkit

VS_ENDPOINT = "genai_lab_guide_vs_endpoint"
VS_INDEX    = f"{CATALOG}.{SCHEMA}.arxiv_chunks_index"

# ── 1. Rebuild retrieval tool from Lab 03 ───────────────────────────────────
vsc   = VectorSearchClient()
index = vsc.get_index(VS_ENDPOINT, VS_INDEX)


def retrieve_context(query: str, num_results: int = 3) -> str:
    results = index.similarity_search(
        query_text=query,
        columns=["chunk_text", "path"],
        num_results=num_results,
        query_type="hybrid",
    )
    docs = results.get("result", {}).get("data_array", [])
    parts = []
    for doc in docs:
        source = doc[1].split("/")[-1].replace(".pdf", "")
        parts.append(f"[Source: {source}]\n{doc[0]}")
    return "\n\n---\n\n".join(parts)


@tool
def search_arxiv_papers(query: str) -> str:
    """Search the arXiv paper corpus for context relevant to the user's question.
    Always call this tool before answering research questions."""
    return retrieve_context(query)


# ── 2. UC function tools via UCFunctionToolkit ───────────────────────────────
client = DatabricksFunctionClient()
uc_toolkit = UCFunctionToolkit(
    function_names=[
        f"{CATALOG}.{SCHEMA}.get_paper_metadata",
        f"{CATALOG}.{SCHEMA}.format_citation",
    ],
    client=client,
)
uc_tools = uc_toolkit.tools

# ── 3. Combine all tools ─────────────────────────────────────────────────────
all_tools = [search_arxiv_papers] + uc_tools
print(f"Tools available: {[t.name for t in all_tools]}")

# ── 4. LLM ──────────────────────────────────────────────────────────────────
llm = ChatDatabricks(
    endpoint=LLM_ENDPOINT,
    max_tokens=1024,
    temperature=0.1,
)

# ── 5. Prompt ────────────────────────────────────────────────────────────────
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are an arXiv Research Assistant with access to three tools: "
        "search_arxiv_papers for retrieving passage content, "
        "get_paper_metadata for looking up paper statistics, and "
        "format_citation for generating properly formatted APA references. "
        "Use the appropriate tool(s) for each question.",
    ),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

# ── 6. Agent ─────────────────────────────────────────────────────────────────
agent          = create_tool_calling_agent(llm, all_tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=all_tools, verbose=True)

# ── 7. Test with a query that exercises multiple tools ───────────────────────
test_query = (
    "How many chunks do we have for the attention paper? "
    "Also retrieve a brief description of the attention mechanism "
    "and format a citation for Vaswani et al. 2017, arXiv:1706.03762."
)

response = agent_executor.invoke({"input": test_query})
print("\n" + "=" * 70)
print(response["output"])

## Key Takeaways

| Concept | What You Did |
|---|---|
| **UC SQL Function** | Created `get_paper_metadata` as a table-valued SQL UDF governed by Unity Catalog |
| **UC Python Function** | Created `format_citation` as a scalar Python UDF with full Python logic |
| **Tool Schemas** | Inspected function metadata via `WorkspaceClient` — the same schema the LLM uses to decide when to call each tool |
| **UCFunctionToolkit** | Converted UC functions into LangChain-compatible tools with a single call |
| **Multi-Tool Agent** | Combined retrieval, SQL, and Python tools so the LLM selects the right tool per sub-task |

### Exam Tips

- UC functions require **`EXECUTE` privilege** on the function and **`USE SCHEMA` / `USE CATALOG`** on the containing schema and catalog.
- SQL UDFs use `RETURNS TABLE` for multi-row results; Python UDFs use `RETURNS <scalar_type> LANGUAGE PYTHON`.
- `UCFunctionToolkit(function_names=[...])` reads UC metadata to build tool definitions — you never write the JSON schema manually.
- The LLM selects tools based on the **function name and `COMMENT`** — write descriptive comments when authoring UC functions that will be used as tools.
- You can test UC functions directly in the **AI Playground** by adding them to the tools panel without writing any agent code.

## Architecture Diagram

![Architecture Diagram](../assets/diagrams/lab-04-uc-functions-agent-tools.png)

## Key Concepts

| Concept | Definition |
|---|---|
| **UC Function** | A SQL or Python function stored in Unity Catalog, versioned, permissioned, and discoverable via the Catalog Explorer |
| **SQL UDF** | A UC function written in Spark SQL using `CREATE FUNCTION ... RETURNS TABLE ... RETURN SELECT ...` — best for set-based operations over Delta tables |
| **Python UDF** | A UC function written in Python using `LANGUAGE PYTHON AS $$ ... $$` — best for scalar transformations and formatting logic |
| **UCFunctionToolkit** | LangChain integration (`unitycatalog-langchain`) that reads UC function schemas and converts them into LangChain `Tool` objects automatically |
| **Tool Schema** | The name, description, and parameter definitions exposed to the LLM so it can decide when and how to call a tool; sourced from the UC function `COMMENT` and parameter types |
| **AI Playground** | Databricks UI for testing Foundation Model endpoints; supports adding UC functions as tools interactively without writing agent code |
| **Tool Calling** | The LLM capability to emit structured function-call requests (name + arguments) that the agent runtime executes and feeds back as observations |

## Exam Preparation

### Exam Practice Questions

**Q1.** Which UC function type should you use to return multiple rows from a Delta table based on an input filter?

- A) `RETURNS STRING LANGUAGE PYTHON`
- B) `RETURNS TABLE (...) RETURN SELECT ...`
- C) `RETURNS INT LANGUAGE SQL`
- D) `RETURNS STRUCT<...>`

**Answer: B** — `RETURNS TABLE` is the SQL UDF syntax for table-valued functions that return multiple rows. Python UDFs return scalar values only.

---

**Q2.** How does an LLM know when to call a UC function that has been added as an agent tool?

- A) The agent hardcodes a routing table mapping intent keywords to tool names
- B) The LLM reads the function's name and `COMMENT` field from the tool schema injected into its context
- C) The agent calls every available tool and picks the best response
- D) The LLM ignores tool metadata and relies solely on few-shot examples in the prompt

**Answer: B** — The LLM uses the tool schema (name + description from the `COMMENT`) to decide which tool matches the user's intent. Well-written comments are essential for correct tool selection.

---

**Q3.** What is the key difference between a UC SQL UDF and a UC Python UDF?

- A) SQL UDFs run on the driver; Python UDFs run on executors
- B) SQL UDFs can only return scalars; Python UDFs can return tables
- C) SQL UDFs use set-based Spark SQL syntax and can return tables; Python UDFs use Python logic and return scalars
- D) Python UDFs require a running cluster; SQL UDFs run serverlessly

**Answer: C** — SQL UDFs (`RETURNS TABLE`) are set-based and leverage Spark SQL optimiser; Python UDFs return scalar values and are evaluated row-by-row.

---

**Q4.** Which class converts Unity Catalog functions into LangChain-compatible tool objects?

- A) `DatabricksFunctionToolkit`
- B) `SparkUDFTool`
- C) `UCFunctionToolkit`
- D) `MLflowFunctionTool`

**Answer: C** — `UCFunctionToolkit` from the `unitycatalog-langchain` package reads UC function metadata and returns a list of LangChain `Tool` objects ready to pass to `create_tool_calling_agent`.

---

**Q5.** When should you prefer a UC function over a LangChain `@tool` for agent tool logic?

- A) When the logic needs to make outbound HTTP calls at runtime
- B) When the tool must return streaming responses
- C) When the logic involves querying Delta tables, needs UC governance (permissions, lineage), or must be shared across multiple agents and notebooks
- D) When you need the tool to run faster than UC function cold-start latency allows

**Answer: C** — UC functions are the right choice when the logic operates on data in the Lakehouse, needs to be governed by Unity Catalog ACLs, or must be reused across teams without code duplication. LangChain `@tool` is better for ephemeral, session-scoped logic or when external API calls are required.

## Cost Breakdown

| Component | Detail | Estimated Cost |
|---|---|---|
| Databricks Serverless compute | UC function creation + notebook execution (~15 min DBU) | ~$0.50 |
| LLM token usage | Multi-tool test query + tool schema injection overhead | ~$0.50 |
| Vector Search queries | Retrieval calls during multi-tool agent test | Included in serverless |
| **Total** | | **~$1–2** |

**Estimated time:** ~25 min | **Estimated cost:** ~$1–2

> Costs vary by workspace region and current DBU pricing. Use the Databricks Cost Dashboard to track actuals.